# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Authenticate to Hugging Face
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

DECISION_MONTH = "2026-03"
START_DATE = "2026-03-01"
END_DATE = "2026-03-31"

FACT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Setup complete.")
print("Decision month:", DECISION_MONTH)
print("Window:", START_DATE, "to", END_DATE)

Setup complete.
Decision month: 2026-03
Window: 2026-03-01 to 2026-03-31


### Distribution notes

The main search-performance fields are expected to be heavy-tailed: a small number of pages can receive very large volumes while many pages receive relatively little traffic. I will inspect the distributions using descriptive statistics and log-scaled summaries before testing signals.

The audit uses observed March 2026 data only.

In [2]:
# Load the March 2026 page-level frame

audit_frame = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS gsc_impressions,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS gsc_clicks,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )
            /
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS gsc_avg_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_sessions, 0)
            ELSE 0
        END
    ) AS ga4_sessions,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN COALESCE(ga4_engaged_sessions, 0)
            ELSE 0
        END
    ) AS ga4_engaged_sessions

FROM read_parquet('{FACT_PATH}')

WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'

GROUP BY
    content_hash_id,
    client_hash_id
""").df()

# Numeric cleanup
numeric_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

for col in numeric_fields:
    audit_frame[col] = pd.to_numeric(
        audit_frame[col],
        errors="coerce"
    )

audit_frame["ctr"] = np.where(
    audit_frame["gsc_impressions"] > 0,
    audit_frame["gsc_clicks"] / audit_frame["gsc_impressions"],
    np.nan
)

print("Audit frame shape:", audit_frame.shape)

distribution_summary = audit_frame[
    numeric_fields
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

display(distribution_summary)

# Log-scaled summaries for heavy-tailed traffic fields
log_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

log_summary = pd.DataFrame({
    col: np.log1p(audit_frame[col].fillna(0))
    for col in log_fields
}).describe().T

print("Log1p summaries:")
display(log_summary)

print(
    "Observation: traffic-like fields are inspected on a log1p scale "
    "because raw web metrics can be heavily right-skewed."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Audit frame shape: (331437, 8)


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
gsc_impressions,331437.0,846.790156,4044.514753,0.0,0.000000,2.000000,216.000000,1707.000000,4225.000000,14909.280000,617124.0
gsc_clicks,331437.0,2.479602,19.651282,0.0,0.000000,0.000000,0.000000,3.000000,11.000000,47.000000,5668.0
gsc_avg_position,176738.0,15.992270,18.097575,0.0,4.917879,8.177966,20.254025,41.053498,58.297297,80.166667,309.0
ga4_sessions,331437.0,3.921735,25.381883,0.0,0.000000,0.000000,1.000000,4.000000,14.000000,85.000000,2730.0
ga4_engaged_sessions,331437.0,0.089160,0.879657,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,224.0


Log1p summaries:


,count,mean,std,min,25%,50%,75%,max
gsc_impressions,331437.0,2.670719,3.093188,0.0,0.0,1.098612,5.379897,13.332827
gsc_clicks,331437.0,0.361636,0.860787,0.0,0.0,0.000000,0.000000,8.642768
ga4_sessions,331437.0,0.454716,0.966219,0.0,0.0,0.000000,0.693147,7.912423
ga4_engaged_sessions,331437.0,0.040632,0.217345,0.0,0.0,0.000000,0.000000,5.416100


Observation: traffic-like fields are inspected on a log1p scale because raw web metrics can be heavily right-skewed.


## 2. Signal test #1 / #2 / #3 (verdict each)

### Signal tests

I will test three observed relationships relevant to content opportunity scoring:

1. Higher search impressions should correspond to greater observed click opportunity.
2. CTR should generally be higher for pages with better observed search position.
3. Higher engagement among GA4 sessions should indicate stronger observed engagement quality.

Each test uses buckets with visible sample sizes and a minimum bucket size of 50 pages.

In [3]:
# ============================================================
# SIGNAL 1 — SEARCH VOLUME VS CLICK OPPORTUNITY
# ============================================================

audit_frame["volume_bucket"] = pd.cut(
    audit_frame["gsc_impressions"],
    bins=[-1, 99, 499, 1999, np.inf],
    labels=[
        "<100",
        "100-499",
        "500-1999",
        "2000+"
    ]
)

volume_test = (
    audit_frame
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

volume_test["clicks_per_page"] = (
    volume_test["total_clicks"]
    / volume_test["n"]
)

volume_test["weighted_ctr"] = np.where(
    volume_test["total_impressions"] > 0,
    volume_test["total_clicks"]
    / volume_test["total_impressions"],
    np.nan
)

print("SIGNAL 1 — Search volume vs click opportunity")

display(
    volume_test[
        [
            "volume_bucket",
            "n",
            "total_impressions",
            "total_clicks",
            "clicks_per_page",
            "weighted_ctr"
        ]
    ]
)

valid_volume = volume_test.dropna(
    subset=["clicks_per_page"]
).copy()

assert (valid_volume["n"] >= 50).all()

if valid_volume["clicks_per_page"].is_monotonic_increasing:
    volume_verdict = "CONFIRMED"
elif valid_volume["clicks_per_page"].is_monotonic_decreasing:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

print("Verdict:", volume_verdict)

SIGNAL 1 — Search volume vs click opportunity


,volume_bucket,n,total_impressions,total_clicks,clicks_per_page,weighted_ctr
0,<100,229996,1861506.0,6457.0,0.028074,0.003469
1,100-499,39517,9879964.0,22626.0,0.572564,0.002290
2,500-1999,32047,33801034.0,92217.0,2.877555,0.002728
3,2000+,29877,235115085.0,700532.0,23.447200,0.002980


Verdict: CONFIRMED


In [4]:
# ============================================================
# SIGNAL 2 — POSITION VS CTR
# ============================================================

audit_frame["position_bucket"] = pd.cut(
    audit_frame["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "20+"
    ],
    include_lowest=True
)

position_test = (
    audit_frame
    .groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

position_test["weighted_ctr"] = np.where(
    position_test["total_impressions"] > 0,
    position_test["total_clicks"]
    / position_test["total_impressions"],
    np.nan
)

position_test["weighted_ctr_pct"] = (
    position_test["weighted_ctr"] * 100
)

print("SIGNAL 2 — Search position vs CTR")

display(
    position_test[
        [
            "position_bucket",
            "n",
            "total_impressions",
            "total_clicks",
            "weighted_ctr_pct"
        ]
    ]
)

valid_position = position_test.dropna(
    subset=["weighted_ctr"]
).copy()

assert (valid_position["n"] >= 50).all()

ctr_values = valid_position["weighted_ctr"].to_numpy()

if len(ctr_values) < 3:
    position_verdict = "MIXED"
else:
    decreasing_pairs = np.sum(
        np.diff(ctr_values) <= 0
    )

    increasing_pairs = np.sum(
        np.diff(ctr_values) > 0
    )

    if decreasing_pairs >= 3:
        position_verdict = "CONFIRMED"
    elif increasing_pairs >= 3:
        position_verdict = "OPPOSITE"
    else:
        position_verdict = "MIXED"

print("Verdict:", position_verdict)

SIGNAL 2 — Search position vs CTR


,position_bucket,n,total_impressions,total_clicks,weighted_ctr_pct
0,1-3,18860,41420140.0,160562.0,0.387642
1,4-5,27712,75455623.0,265413.0,0.351747
2,6-10,55576,72656813.0,215776.0,0.296980
3,11-20,29922,31191659.0,98488.0,0.315751
4,20+,44668,59933354.0,81593.0,0.136140


Verdict: CONFIRMED


In [5]:
# ============================================================
# SIGNAL 3 — GA4 SESSIONS VS ENGAGEMENT RATE
# ============================================================

audit_frame["session_bucket"] = pd.cut(
    audit_frame["ga4_sessions"],
    bins=[-1, 4, 9, 24, 99, np.inf],
    labels=[
        "0-4",
        "5-9",
        "10-24",
        "25-99",
        "100+"
    ]
)

engagement_test = (
    audit_frame
    .groupby("session_bucket", observed=False)
    .agg(
        n=("ga4_sessions", "size"),
        total_sessions=("ga4_sessions", "sum"),
        total_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
    .reset_index()
)

engagement_test["engagement_rate"] = np.where(
    engagement_test["total_sessions"] > 0,
    engagement_test["total_engaged_sessions"]
    / engagement_test["total_sessions"],
    np.nan
)

engagement_test["engagement_rate_pct"] = (
    engagement_test["engagement_rate"] * 100
)

print("SIGNAL 3 — GA4 sessions vs engagement rate")

display(
    engagement_test[
        [
            "session_bucket",
            "n",
            "total_sessions",
            "total_engaged_sessions",
            "engagement_rate_pct"
        ]
    ]
)

valid_engagement = engagement_test.dropna(
    subset=["engagement_rate"]
).copy()

# Only use buckets with enough observations
valid_engagement = valid_engagement[
    valid_engagement["n"] >= 50
].copy()

if len(valid_engagement) < 3:
    engagement_verdict = "FALSE"
else:
    engagement_values = (
        valid_engagement["engagement_rate"]
        .to_numpy()
    )

    increasing = np.sum(
        np.diff(engagement_values) > 0
    )

    decreasing = np.sum(
        np.diff(engagement_values) <= 0
    )

    if increasing >= 3:
        engagement_verdict = "CONFIRMED"
    elif decreasing >= 3:
        engagement_verdict = "OPPOSITE"
    else:
        engagement_verdict = "MIXED"

print("Verdict:", engagement_verdict)

SIGNAL 3 — GA4 sessions vs engagement rate


,session_bucket,n,total_sessions,total_engaged_sessions,engagement_rate_pct
0,0-4,299794,100388.0,2736.0,2.725425
1,5-9,10436,68487.0,2297.0,3.353921
2,10-24,9820,152256.0,4203.0,2.760482
3,25-99,8737,425460.0,10064.0,2.365440
4,100+,2650,553217.0,10251.0,1.852980


Verdict: OPPOSITE


## 3. The flag-linked test

### Flag-linked test — CTR vs position

The flag-linked signal is CTR relative to observed search position. This is connected to the CTR-fix logic discussed in the FlyRank session.

The test checks whether weighted CTR generally falls as observed position becomes worse. I use total clicks divided by total impressions for each position bucket rather than averaging page-level CTRs.

The result is directional evidence only. A confirmed pattern does not prove that changing content will cause CTR to improve.

In [6]:
# ============================================================
# FLAG-LINKED TEST — CTR VS POSITION
# ============================================================

flag_test = position_test.copy()

flag_test["position_order"] = np.arange(
    len(flag_test)
)

print("FLAG-LINKED TEST — CTR vs observed position")

display(
    flag_test[
        [
            "position_bucket",
            "n",
            "total_impressions",
            "total_clicks",
            "weighted_ctr_pct"
        ]
    ]
)

# Sample-size floor
valid_flag_test = flag_test[
    flag_test["n"] >= 50
].dropna(
    subset=["weighted_ctr"]
).copy()

assert len(valid_flag_test) >= 3

flag_ctr = valid_flag_test[
    "weighted_ctr"
].to_numpy()

# For worse positions, CTR should generally decrease.
decreasing_or_flat = np.sum(
    np.diff(flag_ctr) <= 0
)

increasing = np.sum(
    np.diff(flag_ctr) > 0
)

if decreasing_or_flat >= 3:
    flag_verdict = "CONFIRMED"
elif increasing >= 3:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("Flag-linked verdict:", flag_verdict)

if flag_verdict == "CONFIRMED":
    print(
        "The observed bucket pattern supports the directional assumption "
        "behind the CTR-vs-position flag."
    )
elif flag_verdict == "OPPOSITE":
    print(
        "The observed bucket pattern contradicts the assumed CTR-vs-position "
        "relationship; this signal should not be trusted as a rule."
    )
else:
    print(
        "The observed relationship is mixed; the flag should be treated as "
        "decision-support rather than a strong standalone rule."
    )

FLAG-LINKED TEST — CTR vs observed position


,position_bucket,n,total_impressions,total_clicks,weighted_ctr_pct
0,1-3,18860,41420140.0,160562.0,0.387642
1,4-5,27712,75455623.0,265413.0,0.351747
2,6-10,55576,72656813.0,215776.0,0.296980
3,11-20,29922,31191659.0,98488.0,0.315751
4,20+,44668,59933354.0,81593.0,0.136140


Flag-linked verdict: CONFIRMED
The observed bucket pattern supports the directional assumption behind the CTR-vs-position flag.


## 4. What this means in practice

### Practical takeaway

The March 2026 data shows that search-performance signals can provide useful directional evidence for content review, but the relationships are not guarantees of an action outcome. The content team should use volume, CTR, and position as measured signals for prioritization, then validate the individual page and query mix before refreshing it. Any mixed or weak signal should be treated as decision-support rather than an automatic rule.

In [7]:
# ============================================================
# AUDIT SUMMARY
# ============================================================

print("===================================")
print("ML-06 SIGNAL AUDIT SUMMARY")
print("===================================")
print("Decision month:", DECISION_MONTH)
print("Pages audited:", len(audit_frame))
print()
print("Signal 1 — Volume vs click opportunity:", volume_verdict)
print("Signal 2 — Position vs CTR:", position_verdict)
print("Signal 3 — Sessions vs engagement rate:", engagement_verdict)
print("Flag-linked — CTR vs position:", flag_verdict)
print()
print("All tests use observed March 2026 data.")
print("No future-period fields were used.")
print("No label-derived fields were used.")
print("===================================")

ML-06 SIGNAL AUDIT SUMMARY
Decision month: 2026-03
Pages audited: 331437

Signal 1 — Volume vs click opportunity: CONFIRMED
Signal 2 — Position vs CTR: CONFIRMED
Signal 3 — Sessions vs engagement rate: OPPOSITE
Flag-linked — CTR vs position: CONFIRMED

All tests use observed March 2026 data.
No future-period fields were used.
No label-derived fields were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [8]:
# ============================================================
# ML-06 SELF-CHECK
# ============================================================

allowed_verdicts = {
    "CONFIRMED",
    "OPPOSITE",
    "MIXED",
    "FALSE"
}

assert volume_verdict in allowed_verdicts
assert position_verdict in allowed_verdicts
assert engagement_verdict in allowed_verdicts
assert flag_verdict in allowed_verdicts

assert len(audit_frame) > 0

assert (valid_volume["n"] >= 50).all()
assert (valid_position["n"] >= 50).all()
assert (valid_flag_test["n"] >= 50).all()

# No future/label-derived fields
for forbidden in [
    "is_declining_label",
    "trend_direction",
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "future_position",
]:
    assert forbidden not in audit_frame.columns

print("===================================")
print("ML-06 SELF-CHECK")
print("===================================")
print("Decision month:", DECISION_MONTH)
print("Pages audited:", len(audit_frame))
print("Three signal tests completed: YES")
print("Flag-linked test completed: YES")
print("Sample-size floors checked: YES")
print("Future-window leakage: NO")
print("Label-derived leakage: NO")
print("===================================")
print("Self-check passed.")

ML-06 SELF-CHECK
Decision month: 2026-03
Pages audited: 331437
Three signal tests completed: YES
Flag-linked test completed: YES
Sample-size floors checked: YES
Future-window leakage: NO
Label-derived leakage: NO
Self-check passed.
